# Experiment: Repository Exploration

This notebook is the first-pass exploration scaffold for the `microbiome-analysis` repository.

Objectives for this draft:
- inspect the repository structure;
- discover microbiome-relevant data files automatically;
- preview metadata / tabular files when they exist;
- identify missing inputs for downstream analysis;
- define the next cleaning and analysis targets.


## Questions For This Pass

1. What files already exist in this repository?
2. Are there any microbiome-relevant inputs such as metadata tables, abundance tables, FASTA/FASTQ files, or QIIME artifacts?
3. Which files are immediately machine-readable?
4. What standard project folders still need to be created?
5. What should the next notebook focus on after data lands in the repository?


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

plt.style.use("seaborn-v0_8-whitegrid")

REPO_DIR = Path.cwd()
DATA_EXTENSIONS = {
    ".csv", ".tsv", ".txt", ".xlsx", ".xls", ".biom", ".qza", ".qzv",
    ".fastq", ".fq", ".fastq.gz", ".fq.gz", ".fasta", ".fa", ".fna", ".json"
}
IGNORED_PARTS = {".git", ".ipynb_checkpoints", "__pycache__"}

print(f"Repository root: {REPO_DIR}")


## Repository Inventory

This cell inventories the current repository root. If the repository is still nearly empty, that is useful information because it tells us this notebook is serving as a setup scaffold rather than a true data exploration report.


In [ ]:
root_items = []
for path in sorted(REPO_DIR.iterdir()):
    if path.name in IGNORED_PARTS:
        continue
    root_items.append({
        "name": path.name,
        "kind": "directory" if path.is_dir() else (path.suffix.lower() or "<no_ext>"),
        "size_kb": None if path.is_dir() else round(path.stat().st_size / 1024, 2),
    })

root_df = pd.DataFrame(root_items)
root_df


In [ ]:
if root_df.empty:
    print("The repository root is currently empty.")
else:
    display(root_df)
    suffix_summary = (
        root_df.assign(kind=root_df["kind"].fillna("directory"))
        .groupby("kind", dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["count", "kind"], ascending=[False, True])
    )
    display(suffix_summary)


## Recursive File Discovery

The next step is to look recursively for files that commonly appear in microbiome workflows, including metadata tables, abundance tables, FASTA/FASTQ files, and QIIME artifacts.


In [ ]:
def is_relevant_file(path: Path) -> bool:
    suffixes = path.suffixes
    joined = "".join(suffixes[-2:]).lower() if len(suffixes) >= 2 else path.suffix.lower()
    return joined in DATA_EXTENSIONS or path.suffix.lower() in DATA_EXTENSIONS

records = []
for path in sorted(REPO_DIR.rglob("*")):
    if not path.is_file():
        continue
    if any(part in IGNORED_PARTS for part in path.parts):
        continue
    if is_relevant_file(path):
        records.append({
            "relative_path": path.relative_to(REPO_DIR).as_posix(),
            "extension": "".join(path.suffixes[-2:]).lower() if len(path.suffixes) >= 2 else path.suffix.lower(),
            "size_kb": round(path.stat().st_size / 1024, 2),
        })

file_catalog = pd.DataFrame(records)
file_catalog


In [ ]:
if file_catalog.empty:
    print("No microbiome-relevant data files were discovered yet.")
else:
    display(file_catalog)
    display(
        file_catalog.groupby("extension", dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["count", "extension"], ascending=[False, True])
    )


## Tabular File Preview

If the repository already contains CSV / TSV / Excel files, preview one or two of them here to understand their schema before writing cleaning code.


In [ ]:
tabular_extensions = {".csv", ".tsv", ".txt", ".xlsx", ".xls"}

def preview_tabular(path: Path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path).head()
    if suffix in {".tsv", ".txt"}:
        return pd.read_csv(path, sep=None, engine="python").head()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path).head()
    raise ValueError(f"Unsupported preview type: {path}")

preview_candidates = []
if not file_catalog.empty:
    for relative_path in file_catalog["relative_path"]:
        path = REPO_DIR / relative_path
        if path.suffix.lower() in tabular_extensions:
            preview_candidates.append(path)

preview_candidates[:5]


In [ ]:
if not preview_candidates:
    print("No tabular files available for preview yet.")
else:
    for path in preview_candidates[:2]:
        print(f"Preview: {path.relative_to(REPO_DIR).as_posix()}")
        display(preview_tabular(path))


## Suggested Standard Layout

If the repository is still early-stage, the following folder layout is a reasonable target for a microbiome analysis project:

- `data/raw/`: original input tables, metadata, sequencing files, exported artifacts;
- `data/interim/`: cleaned or merged intermediate files;
- `data/processed/`: final analysis-ready matrices;
- `notebooks/`: exploration and analysis notebooks;
- `src/` or `scripts/`: reusable cleaning and plotting code;
- `results/`: figures, tables, and reports.


In [ ]:
expected_paths = [
    REPO_DIR / "data" / "raw",
    REPO_DIR / "data" / "interim",
    REPO_DIR / "data" / "processed",
    REPO_DIR / "notebooks",
    REPO_DIR / "src",
    REPO_DIR / "results",
]

layout_df = pd.DataFrame(
    {
        "path": [p.relative_to(REPO_DIR).as_posix() for p in expected_paths],
        "exists": [p.exists() for p in expected_paths],
    }
)
layout_df


## Notes And Next Steps

What this first notebook should become depends on what lands in the repo next.

Likely next actions:
- add sample metadata and sequencing / abundance tables into `data/raw/`;
- decide whether this repo will focus on amplicon data, metagenomics, or downstream ecological statistics only;
- create a second notebook for schema cleaning once the first real tables are committed;
- move reusable code out of notebooks once parsing rules stabilize.
